# Extract World Population 2024
Source: World Bank — SP.POP.TOTL (API_SP.POP.TOTL_DS2_en_csv_v2_127039.csv)  
Output: `data/country_population.csv` with `alpha_3_code`, `population`, and `year` for 2024.

In [1]:
import pandas as pd
from pathlib import Path
import os

if Path.cwd().name == "extract":
    os.chdir("..")

In [2]:
csv_path = Path("raw") / "API_SP.POP.TOTL_DS2_en_csv_v2_127039" / "API_SP.POP.TOTL_DS2_en_csv_v2_127039.csv"

df = pd.read_csv(csv_path, skiprows=4, encoding="utf-8-sig")

print(df.shape)
df.head(3)

(266, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0,57619.0,58190.0,...,108735.0,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0,NaN,NaN
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0,144904094.0,149033472.0,...,640058741.0,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0,NaN,NaN
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0,9814318.0,10036008.0,...,35688935.0,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0,NaN,NaN


In [3]:
# Keep only Alpha-3 code and 2024 population
pop = df[["Country Code", "2024"]].copy()
pop.columns = ["alpha_3_code", "population"]

# Drop rows with no 2024 figure (aggregates / regions without country-level data)
pop = pop.dropna(subset=["population"])

# Cast population to 64-bit integer (world aggregate exceeds int32 range)
pop["population"] = pop["population"].astype("int64")

# Add year column
pop["year"] = 2024

pop = pop.sort_values("alpha_3_code").reset_index(drop=True)
print(f"{len(pop)} rows")
pop.head(10)

265 rows


,alpha_3_code,population,year
0,ABW,107995,2024
1,AFE,769280888,2024
2,AFG,42647492,2024
3,AFW,521764076,2024
4,AGO,37885849,2024
5,ALB,2377128,2024
6,AND,81938,2024
7,ARB,492612632,2024
8,ARE,10986400,2024
9,ARG,45696159,2024


In [4]:
# Cross-check against country_name_iso_3166.csv to filter to ISO codes only
iso = pd.read_csv("data/country_name_iso_3166.csv")
iso_codes = set(iso["alpha_3_code"])
pop_codes = set(pop["alpha_3_code"])

# Filter population to only ISO codes
pop = pop[pop["alpha_3_code"].isin(iso_codes)].reset_index(drop=True)

only_in_iso = iso_codes - set(pop["alpha_3_code"])

print(f"After filtering to ISO codes: {len(pop)} rows")
print(f"Codes in iso-3166 but NOT in population ({len(only_in_iso)}): {sorted(only_in_iso)[:10]}..." if len(only_in_iso) > 10 else print(f"Codes in iso-3166 but NOT in population ({len(only_in_iso)}): {sorted(only_in_iso)}"))

After filtering to ISO codes: 215 rows
Codes in iso-3166 but NOT in population (34): ['AIA', 'ALA', 'ATA', 'ATF', 'BES', 'BLM', 'BVT', 'CCK', 'COK', 'CXR']...


In [5]:
out_path = Path("data/country_population.csv")
pop.to_csv(out_path, index=False)
print(f"Saved {len(pop)} rows to {out_path}")

Saved 215 rows to data\country_population.csv
